# gRPC 入門チュートリアル

このノートブックでは、gRPC の基本的な仕組みを学びます。

## gRPC とは？

**gRPC** (Google Remote Procedure Call) は、Googleが開発したオープンソースのRPCフレームワークです。

### 主な特徴
- **Protocol Buffers (protobuf)** を使ってインターフェースを定義
- **HTTP/2** をトランスポート層に使用（多重化、ヘッダー圧縮）
- **多言語対応**（Python, Go, Java, C++, Node.js など）
- **4種類の通信パターン**をサポート

```
┌─────────────┐     Protocol Buffers      ┌─────────────┐
│   Client    │ ──── (HTTP/2) ──────────► │   Server    │
│             │ ◄───────────────────────  │             │
└─────────────┘                           └─────────────┘
```

### 4種類の通信パターン
1. **Unary RPC** - 1リクエスト → 1レスポンス（通常の関数呼び出し）
2. **Server Streaming RPC** - 1リクエスト → 複数レスポンス
3. **Client Streaming RPC** - 複数リクエスト → 1レスポンス
4. **Bidirectional Streaming RPC** - 複数リクエスト → 複数レスポンス

## Step 1: 必要なライブラリのインストール

In [ ]:
!pip install grpcio grpcio-tools -q
print("✅ インストール完了")

## Step 2: Protocol Buffers (.proto) ファイルの作成

gRPC では最初に `.proto` ファイルでサービスとメッセージを定義します。

ここでは「挨拶サービス (Greeter)」と「計算サービス (Calculator)」を定義します。

In [ ]:
import os
os.makedirs('grpc_demo', exist_ok=True)

proto_content = '''
syntax = "proto3";

package demo;

// ===== 挨拶サービス =====
service Greeter {
  // Unary RPC: 1リクエスト -> 1レスポンス
  rpc SayHello (HelloRequest) returns (HelloReply);

  // Server Streaming RPC: 1リクエスト -> 複数レスポンス
  rpc SayHelloMultipleTimes (HelloRequest) returns (stream HelloReply);
}

// ===== 計算サービス =====
service Calculator {
  // Unary RPC: 足し算
  rpc Add (CalcRequest) returns (CalcReply);

  // Client Streaming RPC: 複数の数値を受け取り合計を返す
  rpc Sum (stream NumberRequest) returns (CalcReply);

  // Bidirectional Streaming RPC: 数値を受け取るたびに累計を返す
  rpc RunningTotal (stream NumberRequest) returns (stream CalcReply);
}

// メッセージ定義
message HelloRequest {
  string name = 1;
}

message HelloReply {
  string message = 1;
}

message CalcRequest {
  float a = 1;
  float b = 2;
}

message NumberRequest {
  float number = 1;
}

message CalcReply {
  float result = 1;
  string description = 2;
}
'''

with open('grpc_demo/demo.proto', 'w') as f:
    f.write(proto_content)

print("✅ demo.proto を作成しました")
print()
print(proto_content)

## Step 3: protoc でコードを自動生成

`.proto` ファイルからPythonコードを自動生成します。
- `demo_pb2.py` : メッセージクラス
- `demo_pb2_grpc.py` : サービスのスタブ（クライアント）とサービサー（サーバー）

In [ ]:
!python -m grpc_tools.protoc \
    -I grpc_demo \
    --python_out=grpc_demo \
    --grpc_python_out=grpc_demo \
    grpc_demo/demo.proto

print("✅ コード生成完了")
print("生成されたファイル:")
for f in os.listdir('grpc_demo'):
    print(f"  📄 {f}")

## Step 4: サーバーの実装

自動生成されたサービサークラスを継承してサーバーを実装します。

In [ ]:
import sys
sys.path.insert(0, 'grpc_demo')

import grpc
from concurrent import futures
import time
import threading
import demo_pb2
import demo_pb2_grpc

# ===== 挨拶サービスの実装 =====
class GreeterServicer(demo_pb2_grpc.GreeterServicer):

    # Unary RPC
    def SayHello(self, request, context):
        print(f"[Server] SayHello: name={request.name}")
        return demo_pb2.HelloReply(
            message=f"こんにちは、{request.name}さん！"
        )

    # Server Streaming RPC
    def SayHelloMultipleTimes(self, request, context):
        greetings = ["こんにちは", "Hello", "Bonjour", "Hola", "Ciao"]
        for greeting in greetings:
            print(f"[Server] Streaming: {greeting} {request.name}")
            yield demo_pb2.HelloReply(
                message=f"{greeting}、{request.name}さん！"
            )
            time.sleep(0.1)


# ===== 計算サービスの実装 =====
class CalculatorServicer(demo_pb2_grpc.CalculatorServicer):

    # Unary RPC
    def Add(self, request, context):
        result = request.a + request.b
        print(f"[Server] Add: {request.a} + {request.b} = {result}")
        return demo_pb2.CalcReply(
            result=result,
            description=f"{request.a} + {request.b} = {result}"
        )

    # Client Streaming RPC
    def Sum(self, request_iterator, context):
        total = 0
        numbers = []
        for request in request_iterator:
            total += request.number
            numbers.append(request.number)
            print(f"[Server] Sum: 受信 {request.number}, 累計={total}")
        return demo_pb2.CalcReply(
            result=total,
            description=f"合計({' + '.join(map(str, numbers))}) = {total}"
        )

    # Bidirectional Streaming RPC
    def RunningTotal(self, request_iterator, context):
        total = 0
        for request in request_iterator:
            total += request.number
            print(f"[Server] RunningTotal: +{request.number} -> {total}")
            yield demo_pb2.CalcReply(
                result=total,
                description=f"累計: {total}"
            )


# ===== サーバー起動関数 =====
server = None

def start_server(port=50051):
    global server
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    demo_pb2_grpc.add_GreeterServicer_to_server(GreeterServicer(), server)
    demo_pb2_grpc.add_CalculatorServicer_to_server(CalculatorServicer(), server)
    server.add_insecure_port(f'[::]:{port}')
    server.start()
    print(f"✅ gRPC サーバーが起動しました (port: {port})")
    return server

# バックグラウンドでサーバーを起動
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(1)
print("サーバー起動完了！")

## Step 5: クライアントから呼び出す

### 5-1: Unary RPC（最もシンプル）
1つのリクエストを送り、1つのレスポンスを受け取る

In [ ]:
# チャネルとスタブの作成
channel = grpc.insecure_channel('localhost:50051')
greeter_stub = demo_pb2_grpc.GreeterStub(channel)
calc_stub = demo_pb2_grpc.CalculatorStub(channel)

print("=" * 50)
print("【Unary RPC】 1リクエスト → 1レスポンス")
print("=" * 50)

# 挨拶
response = greeter_stub.SayHello(
    demo_pb2.HelloRequest(name="太郎")
)
print(f"[Client] レスポンス: {response.message}")

# 足し算
response = calc_stub.Add(
    demo_pb2.CalcRequest(a=3.5, b=2.5)
)
print(f"[Client] 計算結果: {response.description}")

### 5-2: Server Streaming RPC
1つのリクエストを送り、サーバーから複数のレスポンスをストリームで受け取る

In [ ]:
print("=" * 50)
print("【Server Streaming RPC】 1リクエスト → 複数レスポンス")
print("=" * 50)

responses = greeter_stub.SayHelloMultipleTimes(
    demo_pb2.HelloRequest(name="花子")
)

for i, response in enumerate(responses, 1):
    print(f"[Client] レスポンス {i}: {response.message}")

print("ストリーム終了")

### 5-3: Client Streaming RPC
クライアントから複数のリクエストを送り、最後に1つのレスポンスを受け取る

In [ ]:
print("=" * 50)
print("【Client Streaming RPC】 複数リクエスト → 1レスポンス")
print("=" * 50)

# ジェネレーターで複数リクエストを送信
def number_generator(numbers):
    for n in numbers:
        print(f"[Client] 送信: {n}")
        yield demo_pb2.NumberRequest(number=n)
        time.sleep(0.1)

numbers = [10, 20, 30, 40, 50]
response = calc_stub.Sum(number_generator(numbers))
print(f"[Client] 最終結果: {response.description}")

### 5-4: Bidirectional Streaming RPC
クライアントとサーバーが双方向でストリーミング通信する

In [ ]:
print("=" * 50)
print("【Bidirectional Streaming RPC】 双方向ストリーミング")
print("=" * 50)

def running_total_generator(numbers):
    for n in numbers:
        print(f"[Client] 送信: +{n}")
        yield demo_pb2.NumberRequest(number=n)
        time.sleep(0.1)

numbers = [5, 3, 8, 2, 7]
responses = calc_stub.RunningTotal(running_total_generator(numbers))

for response in responses:
    print(f"[Client] 受信: {response.description}")

print("双方向ストリーム終了")

## Step 6: エラーハンドリング

gRPC には標準的なステータスコードがあります。

In [ ]:
import sys
sys.path.insert(0, 'grpc_demo')
import demo_pb2
import demo_pb2_grpc
import grpc

# エラーを返すサービスを作成して確認
class StrictGreeterServicer(demo_pb2_grpc.GreeterServicer):
    def SayHello(self, request, context):
        if not request.name:
            # INVALID_ARGUMENT エラーを返す
            context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
            context.set_details("名前を入力してください")
            return demo_pb2.HelloReply()

        if request.name == "禁止ユーザー":
            # PERMISSION_DENIED エラーを返す
            context.set_code(grpc.StatusCode.PERMISSION_DENIED)
            context.set_details("このユーザーはアクセスできません")
            return demo_pb2.HelloReply()

        return demo_pb2.HelloReply(message=f"こんにちは、{request.name}さん！")

    def SayHelloMultipleTimes(self, request, context):
        yield demo_pb2.HelloReply(message="test")


# 別ポートで新しいサーバーを起動
from concurrent import futures
import threading, time

strict_server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
demo_pb2_grpc.add_GreeterServicer_to_server(StrictGreeterServicer(), strict_server)
strict_server.add_insecure_port('[::]:50052')
strict_server.start()
time.sleep(0.5)

strict_channel = grpc.insecure_channel('localhost:50052')
strict_stub = demo_pb2_grpc.GreeterStub(strict_channel)

print("=" * 50)
print("【エラーハンドリング】")
print("=" * 50)

# テストケース
test_cases = [
    ("山田", "正常ケース"),
    ("", "空の名前"),
    ("禁止ユーザー", "権限なし"),
]

for name, case in test_cases:
    try:
        response = strict_stub.SayHello(
            demo_pb2.HelloRequest(name=name)
        )
        print(f"✅ [{case}] 成功: {response.message}")
    except grpc.RpcError as e:
        print(f"❌ [{case}] エラー: code={e.code()}, details={e.details()}")

strict_server.stop(0)

## Step 7: メタデータ（ヘッダー）の送受信

gRPC ではリクエストにメタデータ（認証トークンなど）を付与できます。

In [ ]:
class AuthGreeterServicer(demo_pb2_grpc.GreeterServicer):
    def SayHello(self, request, context):
        # メタデータ（ヘッダー）を取得
        metadata = dict(context.invocation_metadata())
        token = metadata.get('authorization', None)

        print(f"[Server] 受信メタデータ: {metadata}")

        if token != 'Bearer secret-token':
            context.set_code(grpc.StatusCode.UNAUTHENTICATED)
            context.set_details("認証トークンが無効です")
            return demo_pb2.HelloReply()

        # レスポンスにメタデータを追加
        context.send_initial_metadata([
            ('x-server-version', '1.0.0'),
        ])

        return demo_pb2.HelloReply(message=f"認証済み: こんにちは、{request.name}さん！")

    def SayHelloMultipleTimes(self, request, context):
        yield demo_pb2.HelloReply(message="test")


auth_server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
demo_pb2_grpc.add_GreeterServicer_to_server(AuthGreeterServicer(), auth_server)
auth_server.add_insecure_port('[::]:50053')
auth_server.start()
time.sleep(0.5)

auth_channel = grpc.insecure_channel('localhost:50053')
auth_stub = demo_pb2_grpc.GreeterStub(auth_channel)

print("=" * 50)
print("【メタデータ（認証）】")
print("=" * 50)

# トークンなし
try:
    response = auth_stub.SayHello(
        demo_pb2.HelloRequest(name="テスト"),
    )
    print(f"✅ 成功: {response.message}")
except grpc.RpcError as e:
    print(f"❌ トークンなし: {e.code()} - {e.details()}")

# 正しいトークン付き
try:
    response, call = auth_stub.SayHello.with_call(
        demo_pb2.HelloRequest(name="田中"),
        metadata=[('authorization', 'Bearer secret-token')]
    )
    print(f"✅ 認証成功: {response.message}")
    print(f"   サーバーメタデータ: {dict(call.initial_metadata())}")
except grpc.RpcError as e:
    print(f"❌ エラー: {e.code()} - {e.details()}")

auth_server.stop(0)

## Step 8: Protocol Buffers の仕組みを理解する

protobuf がどのようにデータをシリアライズするか確認します。

In [ ]:
import json

# メッセージを作成
request = demo_pb2.HelloRequest(name="gRPCテスト")

# protobuf バイナリシリアライズ
serialized = request.SerializeToString()
print("=" * 50)
print("【Protocol Buffers シリアライズ比較】")
print("=" * 50)
print(f"元のデータ: name='{request.name}'")
print(f"")
print(f"[protobuf] バイナリ: {serialized}")
print(f"[protobuf] サイズ: {len(serialized)} bytes")
print()

# JSON と比較
json_data = json.dumps({"name": request.name}, ensure_ascii=False)
json_bytes = json_data.encode('utf-8')
print(f"[JSON]    文字列: {json_data}")
print(f"[JSON]    サイズ: {len(json_bytes)} bytes")
print()

saving = (1 - len(serialized) / len(json_bytes)) * 100
print(f"💡 protobuf は JSON より {saving:.1f}% 小さい")
print()

# デシリアライズ
recovered = demo_pb2.HelloRequest()
recovered.ParseFromString(serialized)
print(f"✅ デシリアライズ結果: name='{recovered.name}'")

## Step 9: まとめ

gRPC の4つの通信パターンと主要な機能を確認しました。

| パターン | リクエスト | レスポンス | 用途例 |
|---------|-----------|-----------|-------|
| **Unary** | 1つ | 1つ | 通常のAPI呼び出し |
| **Server Streaming** | 1つ | 複数 | ログ配信、進捗通知 |
| **Client Streaming** | 複数 | 1つ | ファイルアップロード |
| **Bidirectional** | 複数 | 複数 | チャット、リアルタイム通信 |

### gRPC vs REST API

| 項目 | gRPC | REST |
|------|------|------|
| プロトコル | HTTP/2 | HTTP/1.1 |
| データ形式 | protobuf（バイナリ） | JSON（テキスト） |
| 速度 | 高速 | 比較的遅い |
| 型安全性 | あり（.protoで定義） | なし |
| ストリーミング | ネイティブサポート | 制限あり |
| 可読性 | 低い | 高い |
| ブラウザ対応 | 要プロキシ | ネイティブ |

### 参考リンク
- [gRPC 公式ドキュメント](https://grpc.io/docs/)
- [Protocol Buffers ガイド](https://developers.google.com/protocol-buffers)
- [grpcio-tools PyPI](https://pypi.org/project/grpcio-tools/)

In [ ]:
# サーバーを停止
if server:
    server.stop(0)
    print("✅ サーバーを停止しました")

channel.close()
print("✅ チャネルを閉じました")
print()
print("🎉 チュートリアル完了！")